In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, auc, f1_score, hamming_loss
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in (Path.cwd(), *Path.cwd().parents)
                            if (p / "scripts" / "project.py").exists()) / "scripts"))
from project import *  # ROOT, CACHE_FILE, LEAD_SET, MODELS_DIR, RESULTS_DIR, load_cache, ...
MODELS_DIR, RESULTS_DIR = output_dirs()
X_train, y_train, X_test, y_test, class_names, X_feat_train, X_feat_test = load_cache()
# Lead I model: per-window z-score (see project.CNN_INPUT_NORM); 3-lead: raw mV
X_train, X_test = normalize_cnn_input(X_train), normalize_cnn_input(X_test)
print(f"CNN input normalisation: {CNN_INPUT_NORM}")
tf.keras.utils.set_random_seed(SEED)  # reproducible weights init / shuffling

Lead set  -> lead1 (channels [0])
Signals   -> Train: (19230, 1000, 1) | Test: (2158, 1000, 1)
Features  -> Train: (19230, 24) | Test: (2158, 24)
Targets   -> Classes (5): ['CD', 'HYP', 'MI', 'NORM', 'STTC']
CNN input normalisation: zscore


In [3]:
def build_lightweight_1d_cnn(input_shape=(1000, 3), num_classes=5):
    inputs = layers.Input(shape=input_shape, name="ecg_input")
    
    # Block 1: Initial temporal reduction
    x = layers.Conv1D(16, kernel_size=7, strides=2, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    
    # Block 2: Depthwise Separable Conv
    x = layers.SeparableConv1D(32, kernel_size=5, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    
    # Block 3: Depthwise Separable Conv
    x = layers.SeparableConv1D(64, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    
    # Global Average Pooling replaces parameter-heavy Flatten layers
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="sigmoid", name="class_outputs")(x)
    
    return models.Model(inputs=inputs, outputs=outputs, name="TinyML_1D_CNN")

cnn_1d = build_lightweight_1d_cnn(input_shape=(X_train.shape[1], X_train.shape[2]), num_classes=len(class_names))
cnn_1d.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(multi_label=True, name="macro_auc")]
)
cnn_1d.summary()

cb = [
    callbacks.EarlyStopping(monitor="val_macro_auc", mode="max", patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)
]

print("\nTraining Lightweight 1D-CNN...")
history_1d = cnn_1d.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=25,
    batch_size=64,
    callbacks=cb,
    verbose=1
)

Model: "TinyML_1D_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ ecg_input (InputLayer)          │ (None, 1000, 1)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 500, 16)        │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 500, 16)        │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 250, 16)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv1d                │ (None, 250, 32)        │           624 │
│ (SeparableConv1D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 250, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 125, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv1d_1              │ (None, 125, 64)        │         2,208 │
│ (SeparableConv1D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 125, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ class_outputs (Dense)           │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,653 (22.08 KB)

 Trainable params: 5,429 (21.21 KB)

 Non-trainable params: 224 (896.00 B)


Training Lightweight 1D-CNN...
Epoch 1/25


... (1,402 outputs left out)

258/271 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3748 - macro_auc: 0.8421

263/271 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3751 - macro_auc: 0.8418

268/271 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3748 - macro_auc: 0.8421

271/271 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.3748 - macro_auc: 0.8423 - val_loss: 0.3972 - val_macro_auc: 0.8377 - learning_rate: 1.2500e-04


In [4]:
cnn_1d_probs = cnn_1d.predict(X_test, batch_size=128)
print(f"Lightweight 1D-CNN Macro AUC: {roc_auc_score(y_test, cnn_1d_probs, average='macro'):.4f}")

 1/17 ━━━━━━━━━━━━━━━━━━━━ 1s 102ms/step

12/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step  

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


Lightweight 1D-CNN Macro AUC: 0.8302


In [5]:
import numpy as np

# Save predictions
np.savez_compressed(RESULTS_DIR / "preds_1dcnn.npz", cnn1d=cnn_1d_probs)

# Save the full model architecture and weights
cnn_1d.save(MODELS_DIR / "tinyml_1d_cnn.keras")
print("Saved 1D-CNN predictions and model weights!")

Saved 1D-CNN predictions and model weights!
